In [2]:
# 경로 및 설정
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import clear_output, display
from tqdm.auto import tqdm


REPO_ROOT = Path("/home/stu03/workspace/model")

CONFIG_PATH = (
    REPO_ROOT
    / "configs/experiments/hemorrhage/H-N50.yaml"
)

DATA_ROOT = Path(
    "/home/stu03/workspace/datasets/hemorrhage/v1/"
    "model_ready/nnunet"
)

RESULTS_ROOT = REPO_ROOT / "runs/nnunet"
PYTHON = REPO_ROOT / ".venv/bin/python"

config = yaml.safe_load(
    CONFIG_PATH.read_text(encoding="utf-8")
)

experiment = config["experiment"]
plans = config["plans"]

dataset_name = (
    f"Dataset{int(experiment['dataset_id']):03d}_"
    f"{experiment.get('dataset_name', 'ICHBinary')}"
)

trainer_folder = (
    "nnUNetTrainerICHBaseline"
    f"__{plans['output_identifier']}"
    f"__{experiment['configuration']}"
)

OUTPUT_FOLDER = (
    RESULTS_ROOT
    / dataset_name
    / trainer_folder
    / f"fold_{experiment['fold']}"
)

PROGRESS_PATH = OUTPUT_FOLDER / "progress.json"
METRICS_PATH = OUTPUT_FOLDER / "custom_metrics.json"
LOG_PATH = RESULTS_ROOT / "H-N50-launch.log"
PID_PATH = RESULTS_ROOT / "H-N50.pid"

PLAN_PATH = (
    DATA_ROOT
    / "nnUNet_preprocessed"
    / dataset_name
    / f"{plans['output_identifier']}.json"
)

print("Config:", CONFIG_PATH)
print("Plan:", PLAN_PATH)
print("Output:", OUTPUT_FOLDER)
print("Progress:", PROGRESS_PATH)
print("Log:", LOG_PATH)

Config: /home/stu03/workspace/model/configs/experiments/hemorrhage/H-N50.yaml
Plan: /home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet/nnUNet_preprocessed/Dataset003_ICHBinary2Window/ICHPlansTwoWindowBaselineV1.json
Output: /home/stu03/workspace/model/runs/nnunet/Dataset003_ICHBinary2Window/nnUNetTrainerICHBaseline__ICHPlansTwoWindowBaselineV1__3d_fullres/fold_0
Progress: /home/stu03/workspace/model/runs/nnunet/Dataset003_ICHBinary2Window/nnUNetTrainerICHBaseline__ICHPlansTwoWindowBaselineV1__3d_fullres/fold_0/progress.json
Log: /home/stu03/workspace/model/runs/nnunet/H-N50-launch.log


In [3]:
from copy import deepcopy
from pathlib import Path
import tempfile
import yaml


probe_config = deepcopy(config)

probe_config["experiment"]["name"] = (
    "H-N50-vram-probe"
)

probe_config["trainer"]["epochs"] = 1
probe_config["trainer"]["iterations_per_epoch"] = 1
probe_config["trainer"]["validation_iterations_per_epoch"] = 1
probe_config["trainer"]["save_every"] = 1

probe_config["pruning"]["enabled"] = False

probe_directory = Path(
    tempfile.mkdtemp(prefix="brainon-hn50-vram-")
)

probe_config_path = (
    probe_directory / "H-N50-vram-probe.yaml"
)

probe_results_root = (
    probe_directory / "results"
)

probe_config_path.write_text(
    yaml.safe_dump(
        probe_config,
        allow_unicode=True,
        sort_keys=False,
    ),
    encoding="utf-8",
)

print("Probe config:", probe_config_path)
print("Probe results:", probe_results_root)

Probe config: /tmp/brainon-hn50-vram-ai2q58pn/H-N50-vram-probe.yaml
Probe results: /tmp/brainon-hn50-vram-ai2q58pn/results


In [4]:
import os
import subprocess
import sys


probe_code = r"""
import json
import torch

from brainon_ai.nnunet.runner import run_from_config


torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

result = {}

try:
    output = run_from_config(
        config_path=__CONFIG_PATH__,
        data_root=__DATA_ROOT__,
        results_root=__RESULTS_ROOT__,
        run_validation=False,
    )

    result["status"] = "completed"
    result["output"] = str(output)

except torch.cuda.OutOfMemoryError as exc:
    result["status"] = "out_of_memory"
    result["error"] = str(exc)

except Exception as exc:
    result["status"] = "failed"
    result["error"] = repr(exc)

finally:
    result["peak_allocated_gb"] = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )
    result["peak_reserved_gb"] = (
        torch.cuda.max_memory_reserved()
        / 1024**3
    )

    free_bytes, total_bytes = (
        torch.cuda.mem_get_info()
    )

    result["free_gb_after"] = (
        free_bytes / 1024**3
    )
    result["total_gb"] = (
        total_bytes / 1024**3
    )

    print(
        "VRAM_PROBE_RESULT="
        + json.dumps(result)
    )
"""

probe_code = (
    probe_code
    .replace(
        "__CONFIG_PATH__",
        repr(str(probe_config_path)),
    )
    .replace(
        "__DATA_ROOT__",
        repr(str(DATA_ROOT)),
    )
    .replace(
        "__RESULTS_ROOT__",
        repr(str(probe_results_root)),
    )
)

probe_environment = os.environ.copy()

probe_environment["nnUNet_n_proc_DA"] = "0"
probe_environment["nnUNet_def_n_proc"] = "1"

probe_environment["PYTHONPATH"] = (
    os.pathsep.join(
        [
            str(REPO_ROOT / "src"),
            probe_environment.get(
                "PYTHONPATH",
                "",
            ),
        ]
    ).rstrip(os.pathsep)
)

probe = subprocess.run(
    [
        str(PYTHON),
        "-c",
        probe_code,
    ],
    cwd=REPO_ROOT,
    env=probe_environment,
    text=True,
    capture_output=True,
)

print(probe.stdout)

if probe.stderr:
    print("STDERR:")
    print(probe.stderr[-5000:])

print("Return code:", probe.returncode)

Trainer 'nnUNetTrainerICHBaseline' not found in nnunetv2.training.nnUNetTrainer.
Searching in external trainer paths from environment variable 'nnUNet_extTrainer'...
Searching in: /home/stu03/workspace/model/src/brainon_ai/nnunet/trainers
Searching for class nnUNetTrainerICHBaseline in folder /home/stu03/workspace/model/src/brainon_ai/nnunet/trainers with current module None
  Inspecting module: configurable_trainer
Found class nnUNetTrainerICHBaseline in configurable_trainer
Using trainer 'nnUNetTrainerICHBaseline' from: /home/stu03/workspace/model/src/brainon_ai/nnunet/trainers
Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
##############################################################

In [5]:
# 실행 전 검사
required_paths = [
    CONFIG_PATH,
    DATA_ROOT / "nnUNet_raw",
    DATA_ROOT / "nnUNet_preprocessed",
    PLAN_PATH,
]

missing = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "필수 경로가 없습니다:\n"
        + "\n".join(missing)
    )

assert config["network"]["name"] == "default"
assert plans["base_identifier"] == "nnUNetPlans"
assert plans["output_identifier"] == "ICHPlansTwoWindowBaselineV1"


raw_dataset = DATA_ROOT / "nnUNet_raw" / dataset_name
dataset_json = json.loads((raw_dataset / "dataset.json").read_text(encoding="utf-8"))
assert dataset_json["channel_names"] == {"0": "brain_window", "1": "subdural_window"}
channel_0 = sorted((raw_dataset / "imagesTr").glob("*_0000.nii.gz"))
channel_1 = sorted((raw_dataset / "imagesTr").glob("*_0001.nii.gz"))
assert len(channel_0) == len(channel_1) == int(dataset_json["numTraining"])
print("2채널 train cases:", len(channel_0))

print("H-N50 실행 준비 완료")

2채널 train cases: 226
H-N50 실행 준비 완료


In [6]:
# dry run
environment = os.environ.copy()

environment["nnUNet_n_proc_DA"] = "0"
environment["nnUNet_def_n_proc"] = "1"

environment["PYTHONPATH"] = os.pathsep.join(
    [
        str(REPO_ROOT / "src"),
        environment.get("PYTHONPATH", ""),
    ]
).rstrip(os.pathsep)

command = [
    str(PYTHON),
    "-m",
    "brainon_ai.nnunet.runner",
    "--config",
    str(CONFIG_PATH),
    "--data-root",
    str(DATA_ROOT),
    "--results-root",
    str(RESULTS_ROOT),
    "--dry-run",
]

result = subprocess.run(
    command,
    cwd=REPO_ROOT,
    env=environment,
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(
        f"dry-run 실패: returncode={result.returncode}"
    )

{
  "config": "/home/stu03/workspace/model/configs/experiments/hemorrhage/H-N50.yaml",
  "data_paths": {
    "nnUNet_raw": "/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet/nnUNet_raw",
    "nnUNet_preprocessed": "/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet/nnUNet_preprocessed",
    "nnUNet_results": "/home/stu03/workspace/model/runs/nnunet"
  },
  "output_folder": "/home/stu03/workspace/model/runs/nnunet/Dataset003_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansTwoWindowBaselineV1__3d_fullres/fold_0",
  "experiment": {
    "name": "H-N50-nnunet-two-window-baseline",
    "dataset_id": 3,
    "dataset_name": "ICHBinary2Window",
    "configuration": "3d_fullres",
    "fold": 0,
    "device": "auto",
    "continue_training": false,
    "validation_checkpoint": "best_composite"
  },
  "plans": {
    "base_identifier": "nnUNetPlans",
    "output_identifier": "ICHPlansTwoWindowBaselineV1",
    "patch_size": [
      20,
      320,
      256
    ],
    "batch_si

In [7]:
# 기존 실행 충돌 및 resume 확인
config = yaml.safe_load(
    CONFIG_PATH.read_text(encoding="utf-8")
)

continue_training = bool(
    config["experiment"].get(
        "continue_training",
        False,
    )
)

print("continue_training:", continue_training)
print("output folder:", OUTPUT_FOLDER)

if OUTPUT_FOLDER.exists():
    existing_files = sorted(
        path.name
        for path in OUTPUT_FOLDER.iterdir()
    )

    print("기존 output 폴더가 있습니다.")
    print(existing_files)

    latest_checkpoint = (
        OUTPUT_FOLDER / "checkpoint_latest.pth"
    )

    final_checkpoint = (
        OUTPUT_FOLDER / "checkpoint_final.pth"
    )

    print(
        "checkpoint_latest:",
        latest_checkpoint.exists(),
    )
    print(
        "checkpoint_final:",
        final_checkpoint.exists(),
    )

    if continue_training:
        if not latest_checkpoint.exists():
            raise RuntimeError(
                "continue_training=true이지만 "
                "checkpoint_latest.pth가 없습니다."
            )

        print(
            "재개 가능한 checkpoint를 확인했습니다."
        )

    else:
        raise RuntimeError(
            "기존 H-N50 결과가 있습니다. "
            "새 학습이면 결과 폴더를 삭제하고, "
            "재개하려면 H-N50.yaml에서 "
            "continue_training: true로 바꾸세요."
        )

else:
    if continue_training:
        raise RuntimeError(
            "continue_training=true이지만 "
            "기존 H-N50 결과 폴더가 없습니다."
        )

    print(
        "기존 결과와 충돌하지 않습니다. "
        "새 학습을 시작할 수 있습니다."
    )

continue_training: False
output folder: /home/stu03/workspace/model/runs/nnunet/Dataset003_ICHBinary2Window/nnUNetTrainerICHBaseline__ICHPlansTwoWindowBaselineV1__3d_fullres/fold_0
기존 결과와 충돌하지 않습니다. 새 학습을 시작할 수 있습니다.


In [8]:
# H-N50 학습
RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

train_command = [
    str(PYTHON),
    "-m",
    "brainon_ai.nnunet.runner",
    "--config",
    str(CONFIG_PATH),
    "--data-root",
    str(DATA_ROOT),
    "--results-root",
    str(RESULTS_ROOT),
]

log_handle = LOG_PATH.open(
    "a",
    encoding="utf-8",
    buffering=1,
)

log_handle.write(
    "\n\n===== H-N50 launch =====\n"
)
log_handle.write(
    " ".join(train_command) + "\n"
)
log_handle.flush()

process = subprocess.Popen(
    train_command,
    cwd=REPO_ROOT,
    env=environment,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    text=True,
    start_new_session=True,
)

PID_PATH.write_text(
    str(process.pid),
    encoding="utf-8",
)

print("H-N50 학습 시작")
print("PID:", process.pid)
print("Log:", LOG_PATH)
print("Progress:", PROGRESS_PATH)

H-N50 학습 시작
PID: 1008692
Log: /home/stu03/workspace/model/runs/nnunet/H-N50-launch.log
Progress: /home/stu03/workspace/model/runs/nnunet/Dataset003_ICHBinary2Window/nnUNetTrainerICHBaseline__ICHPlansTwoWindowBaselineV1__3d_fullres/fold_0/progress.json


In [30]:
# 프로세스 즉시 확인
time.sleep(3)

return_code = process.poll()

if return_code is None:
    print("학습 프로세스 실행 중")
else:
    print(
        f"학습 프로세스 조기 종료: {return_code}"
    )

    if LOG_PATH.exists():
        lines = LOG_PATH.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()

        print("\n".join(lines[-80:]))

학습 프로세스 실행 중


In [9]:
# 실시간 진행률 표시
total_epochs = int(
    config["trainer"]["epochs"]
)

terminal_statuses = {
    "completed",
    "early_stopped",
    "failed",
}

bar = tqdm(
    total=total_epochs,
    desc="H-N50 epochs",
    unit="epoch",
)

last_epoch = 0

while True:
    state = {}

    if PROGRESS_PATH.exists():
        try:
            state = json.loads(
                PROGRESS_PATH.read_text(
                    encoding="utf-8"
                )
            )
        except json.JSONDecodeError:
            time.sleep(1)
            continue

        current_epoch = int(
            state.get("current_epoch", 0)
        )

        if current_epoch > last_epoch:
            bar.update(
                current_epoch - last_epoch
            )
            last_epoch = current_epoch

        bar.set_postfix(
            status=state.get("status"),
            dice=state.get("pseudo_dice"),
            recall=state.get("pseudo_recall"),
            composite=state.get("composite"),
            best_dice=state.get("best_dice"),
            best_composite=state.get(
                "best_composite"
            ),
            bad_checks=state.get(
                "pruning_bad_checks"
            ),
            patience=state.get(
                "pruning_patience"
            ),
        )

    # 현재 커널에서 시작한 프로세스가 있으면 상태 확인
    process_return_code = None

    if "process" in globals():
        process_return_code = process.poll()

        if process_return_code is not None:
            break

    status = state.get("status")

    if status in terminal_statuses:
        break

    clear_output(wait=True)

    display(
        {
            "status": status or "starting",
            "epoch": (
                f"{state.get('current_epoch', 0)}"
                f"/{state.get('total_epochs', total_epochs)}"
            ),
            "progress_percent": state.get(
                "progress_percent", 0
            ),
            "eta_seconds": state.get(
                "eta_seconds"
            ),
            "pseudo_dice": state.get(
                "pseudo_dice"
            ),
            "pseudo_recall": state.get(
                "pseudo_recall"
            ),
            "composite": state.get(
                "composite"
            ),
            "best_dice": state.get(
                "best_dice"
            ),
            "best_recall": state.get(
                "best_recall"
            ),
            "best_composite": state.get(
                "best_composite"
            ),
            "early_stopping": (
                f"{state.get('pruning_bad_checks', 0)}"
                f"/{state.get('pruning_patience', 30)}"
            ),
            "checkpoint_latest": (
                OUTPUT_FOLDER
                / "checkpoint_latest.pth"
            ).exists(),
        }
    )

    time.sleep(5)

bar.close()

if "process" in globals():
    print(
        "process return code:",
        process.poll(),
    )

if PROGRESS_PATH.exists():
    final_state = json.loads(
        PROGRESS_PATH.read_text(
            encoding="utf-8"
        )
    )

    display(final_state)

{'status': 'running',
 'epoch': '234/300',
 'progress_percent': 78.0,
 'eta_seconds': 1234.8757013602135,
 'pseudo_dice': 0.814295768737793,
 'pseudo_recall': 0.8156048059463501,
 'composite': 0.8149502873420715,
 'best_dice': 0.8528432250022888,
 'best_recall': 0.8629496097564697,
 'best_composite': 0.8498735427856445,
 'early_stopping': '29/30',
 'checkpoint_latest': True}

H-N50 epochs:  78%|███████▊  | 235/300 [1:15:16<20:49, 19.22s/epoch, bad_checks=30, best_composite=0.85, best_dice=0.853, composite=None, dice=None, patience=30, recall=None, status=early_stopped]

process return code: None


{'status': 'early_stopped',
 'message': None,
 'updated_at': '2026-09-01T18:16:50.409465+00:00',
 'current_epoch': 235,
 'total_epochs': 300,
 'progress_percent': 78.33,
 'session_elapsed_seconds': 4519.3,
 'last_epoch_seconds': 18.328107118606567,
 'mean_epoch_seconds': 18.70861181096828,
 'eta_seconds': 1216.059767712938,
 'pseudo_dice': None,
 'pseudo_recall': None,
 'composite': None,
 'best_dice': 0.8528432250022888,
 'best_recall': 0.8629496097564697,
 'best_composite': 0.8498735427856445,
 'pruning_bad_checks': 30,
 'pruning_patience': 30,
 'output_folder': '/home/stu03/workspace/model/runs/nnunet/Dataset003_ICHBinary2Window/nnUNetTrainerICHBaseline__ICHPlansTwoWindowBaselineV1__3d_fullres/fold_0',
 'batch_size': 1,
 'patch_size': [20, 320, 256]}

In [10]:
# 로그 마지막 부분 확인
if not LOG_PATH.exists():
    print("로그 파일이 아직 없습니다.")
else:
    lines = LOG_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()

    print("\n".join(lines[-100:]))

2026-09-01 18:13:20.793351: Epoch 224
2026-09-01 18:13:20.793417: Current learning rate: 0.00291
2026-09-01 18:13:39.531113: ICH metrics: pseudo_dice=0.8208, pseudo_recall=0.8018, composite=0.8113
2026-09-01 18:13:39.531284: train_loss -0.4112
2026-09-01 18:13:39.531316: val_loss -0.4721
2026-09-01 18:13:39.531337: Pseudo dice [np.float32(0.8208)]
2026-09-01 18:13:39.531370: Epoch time: 18.74 s
2026-09-01 18:13:39.701736: Yayy! New best EMA pseudo Dice: 0.7889000177383423
2026-09-01 18:13:40.258604: 
2026-09-01 18:13:40.258660: Epoch 225
2026-09-01 18:13:40.258727: Current learning rate: 0.00287
2026-09-01 18:13:58.902258: ICH metrics: pseudo_dice=0.7644, pseudo_recall=0.6697, composite=0.7170
2026-09-01 18:13:58.902397: train_loss -0.4416
2026-09-01 18:13:58.902427: val_loss -0.4157
2026-09-01 18:13:58.902446: Pseudo dice [np.float32(0.7644)]
2026-09-01 18:13:58.902479: Epoch time: 18.64 s
2026-09-01 18:13:59.297830: 
2026-09-01 18:13:59.297885: Epoch 226
2026-09-01 18:13:59.297952: C

In [14]:
# 학습 후 결과 확인
if METRICS_PATH.exists():
    metrics = json.loads(
        METRICS_PATH.read_text(
            encoding="utf-8"
        )
    )

    print("Patch-based online metrics")
    display(metrics)
else:
    print(
        "custom_metrics.json이 아직 없습니다."
    )

validation_summary = (
    OUTPUT_FOLDER
    / "validation"
    / "summary.json"
)

if validation_summary.exists():
    summary = json.loads(
        validation_summary.read_text(
            encoding="utf-8"
        )
    )

    print("Full-volume validation")
    display(summary["foreground_mean"])
else:
    print(
        "Full-volume validation summary가 "
        "아직 없습니다."
    )

Patch-based online metrics


{'epoch': 234,
 'pseudo_dice': 0.7884504199028015,
 'pseudo_recall': 0.823082447052002,
 'composite': 0.8057664334774017,
 'best_dice': 0.8528432250022888,
 'best_recall': 0.8629496097564697,
 'best_composite': 0.8498735427856445,
 'pruning_best': 0.8498735427856445,
 'pruning_bad_checks': 30,
 'metrics_note': 'patch-based online validation metrics; final reporting requires full-volume evaluation'}

Full-volume validation


{'Dice': 0.5122925316134346,
 'FN': 5542.325,
 'FP': 5920.875,
 'IoU': 0.41579949609599476,
 'TN': 8889205.675,
 'TP': 18780.725,
 'n_pred': 24701.6,
 'n_ref': 24323.05}

In [15]:
# H-N00 vs H-N50 실제 validation 비교
BASELINE_SUMMARY = REPO_ROOT / "runs/nnunet/Dataset001_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansBaselineV1__3d_fullres/fold_0/validation/summary.json"
H_N50_SUMMARY = OUTPUT_FOLDER / "validation/summary.json"

def summary_rows(path, experiment_name):
    payload = json.loads(path.read_text(encoding="utf-8"))
    rows = []
    for item in payload["metric_per_case"]:
        name = item["reference_file"].replace("\\", "/").rsplit("/", 1)[-1]
        metric = item["metrics"]["1"]
        source = "BHSD" if name.startswith("BHSD_") else "PhysioNet"
        rows.append({"experiment": experiment_name, "case_id": name.removesuffix(".nii.gz"), "source": source, "dice": float(metric["Dice"]), "n_ref": int(metric["n_ref"])})
    return pd.DataFrame(rows)

if BASELINE_SUMMARY.is_file() and H_N50_SUMMARY.is_file():
    comparison_cases = pd.concat([summary_rows(BASELINE_SUMMARY, "H-N00 single-window"), summary_rows(H_N50_SUMMARY, "H-N50 two-window")], ignore_index=True)
    positive_cases = comparison_cases[comparison_cases.n_ref > 0]
    display(positive_cases.groupby(["experiment", "source"]).agg(cases=("case_id", "size"), macro_dice=("dice", "mean"), median_dice=("dice", "median")).reset_index())

    CASE_STATS = REPO_ROOT / "runs/analysis/bhsd_physionet_domain_comparison/case_statistics.csv"
    if CASE_STATS.is_file():
        volumes = pd.read_csv(CASE_STATS)[["case_id", "hemorrhage_volume_ml"]]
        by_volume = positive_cases.merge(volumes, on="case_id", how="left")
        labels = ["<1 mL", "1–5 mL", "5–20 mL", "≥20 mL"]
        by_volume["volume_group"] = pd.cut(by_volume.hemorrhage_volume_ml, [0, 1, 5, 20, float("inf")], labels=labels, right=False)
        volume_summary = by_volume.groupby(["experiment", "source", "volume_group"], observed=False).agg(cases=("case_id", "size"), macro_dice=("dice", "mean")).reset_index()
        display(volume_summary)
        focus = by_volume[by_volume.case_id.isin(["PHYS_58", "PHYS_82", "PHYS_78"])].sort_values(["case_id", "experiment"])
        print("작은 PhysioNet 핵심 케이스")
        display(focus[["experiment", "case_id", "hemorrhage_volume_ml", "dice"]])
    else:
        print("10번 노트북을 먼저 실행하면 부피 구간별 비교도 표시됩니다:", CASE_STATS)
else:
    print("H-N50 validation 완료 후 실행하세요.")
    print("H-N00:", BASELINE_SUMMARY.is_file(), "H-N50:", H_N50_SUMMARY.is_file())


,experiment,source,cases,macro_dice,median_dice
0,H-N00 single-window,BHSD,28,0.624656,0.666136
1,H-N00 single-window,PhysioNet,7,0.474236,0.652693
2,H-N50 two-window,BHSD,28,0.606724,0.679015
3,H-N50 two-window,PhysioNet,7,0.500488,0.704472


,experiment,source,volume_group,cases,macro_dice
0,H-N00 single-window,BHSD,<1 mL,2,0.188090
1,H-N00 single-window,BHSD,1–5 mL,7,0.487943
2,H-N00 single-window,BHSD,5–20 mL,5,0.530359
3,H-N00 single-window,BHSD,≥20 mL,14,0.789057
4,H-N00 single-window,PhysioNet,<1 mL,1,0.000000
5,H-N00 single-window,PhysioNet,1–5 mL,1,0.000000
6,H-N00 single-window,PhysioNet,5–20 mL,3,0.583290
7,H-N00 single-window,PhysioNet,≥20 mL,2,0.784890
8,H-N50 two-window,BHSD,<1 mL,2,0.076415
9,H-N50 two-window,BHSD,1–5 mL,7,0.432962


작은 PhysioNet 핵심 케이스


,experiment,case_id,hemorrhage_volume_ml,dice
29,H-N00 single-window,PHYS_58,0.252899,0.000000
64,H-N50 two-window,PHYS_58,0.252899,0.000000
32,H-N00 single-window,PHYS_78,5.720863,0.149406
67,H-N50 two-window,PHYS_78,5.720863,0.241678
33,H-N00 single-window,PHYS_82,1.694899,0.000000
68,H-N50 two-window,PHYS_82,1.694899,0.000000
